In [35]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [37]:
# 라이브러리 설치
!pip install faster-whisper rapidfuzz g2pk konlpy python-mecab-ko

In [38]:
import os, re, json, glob, csv, random, glob
from dataclasses import dataclass
from typing import Dict, List, Optional, Any, Tuple
from g2pk import G2p
from faster_whisper import WhisperModel
from rapidfuzz.distance import Levenshtein
from rapidfuzz import process, fuzz
from mecab import MeCab

In [39]:
# 드라이브 내 프로젝트 폴더로 이동 (본인의 경로에 맞게 수정하세요)
%cd /content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/seowonryeol
# 데이터 경로 설정 (Colab 환경 기준)
AUDIO_FOLDER      = os.environ.get("AUDIO_FOLDER", "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/voice_record/seowonryeol")            # 평가할 음성 파일(.wav) 폴더
TRANSCRIPTS_PATH  = os.environ.get("TRANSCRIPTS_PATH", "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/seowonryeol/transcripts.json") # 정답지 (Ground Truth)
BIAS_PATH         = os.environ.get("BIASING_LIST_PATH", "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/seowonryeol/biasing_list.json") # ASR 힌트 단어 리스트
EPG_PATH          = os.environ.get("EPG_PATH", "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/seowonryeol/epg.json")                 # 실시간 방송 편성표 DB
CATALOG_PATH      = os.environ.get("CATALOG_PATH", "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/seowonryeol/catalog.json")         # VOD 영화/드라마 목록 DB

# 실험 변수 (Hyper-parameters)
TOPK_SWEEP = [20]             # Hotwords 개수 변화 실험 (0개일 때 vs 20개일 때 인식률 차이 비교)
REPEAT_NUM=5 #각 TOPK_SWEEP별 편차를 줄이기 위한 반복횟수
POSTPROCESS_SWEEP = [0]           # 후처리 적용 여부 (0: OFF, 1: ON)
WL_TOPN    = int(os.environ.get("WL_TOPN", "80"))    # 교정 시 검색할 후보 단어의 최대 개수

# [후처리 안전장치] 과교정(Over-correction) 방지 Gate 파라미터
# 설명: ASR 결과가 '무한도전'인데 DB에 '무한도전 레전드'가 있다고 무조건 바꾸면 안 됨.
#       두 단어 간의 유사도가 아래 임계값을 넘어야만 교정을 수행함.
RULE_WRATIO_TH = int(os.environ.get("RULE_WRATIO_TH", "92")) # 문자열 유사도(0~100)가 92점 이상이어야 함
RULE_GATE      = float(os.environ.get("RULE_GATE", "0.34"))  # 부분 오타율이 34% 이내여야 함 (너무 다르면 다른 단어로 판단)
RULE_TOL       = int(os.environ.get("RULE_TOL", "2"))        # 글자 수 차이 허용 범위

# ASR 모델 하드웨어 설정
ASR_MODEL   = os.environ.get("ASR_MODEL", "medium")
ASR_DEVICE  = os.environ.get("ASR_DEVICE", "cuda")     # GPU 사용 (없으면 cpu)
ASR_COMPUTE = os.environ.get("ASR_COMPUTE", "float32")
ASR_LANG    = os.environ.get("ASR_LANG", "ko")         # 한국어 설정
ASR_BEAM    = int(os.environ.get("ASR_BEAM", "5"))     # 탐색 폭 (클수록 정확하나 느림)

# 결과 저장 경로
OUT_ROWS = "./asr_detail_wonryeol_bias_update.csv"     # 상세 로그: 파일 하나하나의 인식 결과 및 점수
OUT_SUM  = "./asr_summary_wonryeol_bias_update.csv"  # 요약 로그: 실험 설정별 평균 점수 (보고서용 표 제작에 사용)

/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/seowonryeol


In [40]:

# -----------------------------
# 1) 텍스트 정규화 및 평가 함수 (Metrics)
# -----------------------------
# 설명: 컴퓨터가 채점하기 좋게 텍스트를 다듬고, 점수를 매기는 함수들입니다.


CHAR_TO_KO = {
    # 알파벳 발음 (숫자는 아래 num_to_ko에서 처리하므로 여기서는 제외해도 됩니다)
    "a": "에이", "b": "비", "c": "씨", "d": "디", "e": "이", "f": "에프", "g": "지",
    "h": "에이치", "i": "아이", "j": "제이", "k": "케이", "l": "엘", "m": "엠",
    "n": "엔", "o": "오", "p": "피", "q": "큐", "r": "알", "s": "에스",
    "t": "티", "u": "유", "v": "브이", "w": "더블유", "x": "엑스", "y": "와이", "z": "제트"
}

NUM_SENSE_MAP = {"하나":"일", "한":"일", "둘":"이", "두":"이", "셋":"삼", "세":"삼", "여덟":"팔", "열":"십"}

NUM_KO_0_10 = {
    "영": 0, "공": 0, "일": 1, "하나": 1, "한": 1, "이": 2, "둘": 2, "두": 2,
    "삼": 3, "셋": 3, "세": 3, "사": 4, "넷": 4, "네": 4, "오": 5, "다섯": 5,
    "육": 6, "여섯": 6, "칠": 7, "일곱": 7, "팔": 8, "여덟": 8, "구": 9, "아홉": 9, "십": 10,
}


NUM_VARIATION_MAP = {"하나":"일", "한":"일", "둘":"이", "두":"이", "셋":"삼", "세":"삼", "네":"사", "여덟":"팔", "열":"십"}


def num_to_ko(num_str: str) -> str:
    """ 숫자를 한국어 읽기 방식(0~999)으로 변환 """
    try:
        n = int(num_str)
        if n == 0: return "영"
        units = ["", "일", "이", "삼", "사", "오", "육", "칠", "팔", "구"]
        tens = ["", "십", "이십", "삼십", "사십", "오십", "육십", "칠십", "팔십", "구십"]
        hundreds = ["", "백", "이백", "삼백", "사백", "오백", "육백", "칠백", "팔백", "구백"]

        if n >= 100:
            h_val, rest = divmod(n, 100)
            t_val, u_val = divmod(rest, 10)
            t_str = tens[t_val]
            if t_val == 1: t_str = "십" # '일십' 방지
            return hundreds[h_val] + t_str + units[u_val]
        elif n >= 10:
            t_val, u_val = divmod(n, 10)
            t_str = tens[t_val]
            if t_val == 1: t_str = "십"
            return t_str + units[u_val]
        else:
            return units[n]
    except: return num_str

def num_to_ko(num_str: str) -> str:
    """ 숫자를 한국어 읽기 방식(0~999)으로 변환 """
    try:
        n = int(num_str)
        if n == 0: return "영"
        u, t, h = ["","일","이","삼","사","오","육","칠","팔","구"], ["","십","이십","삼십","사십","오십","육십","칠십","팔십","구십"], ["","백","이백","삼백","사백","오백","육백","칠백","팔백","구백"]
        if n >= 100:
            hv, r = divmod(n, 100); tv, uv = divmod(r, 10)
            return h[hv] + (t[tv] if tv != 1 else "십") + u[uv]
        elif n >= 10:
            tv, uv = divmod(n, 10)
            return (t[tv] if tv != 1 else "십") + u[uv]
        return u[n]
    except: return num_str

def norm_ko(text: str, remove_space: bool = False) -> str:
    if not text: return ""

    # 1. 소문자화 및 기본 정돈
    s = str(text).lower().strip()

    # 3. 아라비아 숫자 -> 한글 발음 (예: "30" -> "삼십")
    s = re.sub(r'\d+', lambda m: num_to_ko(m.group()), s)

    # 4. 고유어 수사 -> 한자어 수사 통일 (예: "두 명" -> "이 명")
    # \b를 사용하여 '열고', '이상한' 등의 단어 내부 글자 변환 방지
    for k, v in NUM_SENSE_MAP.items():
        s = re.sub(rf'\b{k}\b', v, s)

    # 5. 개별 알파벳 처리 (남은 영어 발음화)
    for eng, ko in CHAR_TO_KO.items():
        s = s.replace(eng, ko)

    # 6. g2pk 투입 (최종 발음 기호화)
    # 채점 시 맞춤법보다 '소리'의 일치도를 보기 위함입니다.
    # s = g2p(s)

    # 7. 특수문자 제거 및 공백 처리
    if remove_space:
        s = re.sub(r"[^0-9\uac00-\ud7a3]", "", s)
    else:
        s = re.sub(r"[^0-9\uac00-\ud7a3\s]", "", s)
        s = re.sub(r"\s+", " ", s).strip()
    return s

def cer_norm(ref: str, hyp: str) -> float:
    """
    [Normalized CER] 공백을 무시하고 순수 한글 발음의 일치도를 측정합니다.
    """
    r = norm_ko(ref, remove_space=True)
    h = norm_ko(hyp, remove_space=True)

    if not r:
        return 0.0 if not h else 1.0

    # 문자 단위 편집 거리 계산
    return Levenshtein.distance(r, h) / len(r)

def wer_norm(ref: str, hyp: str) -> float:
    """
    형태소 기반 WER: 띄어쓰기 오차를 무시하고
    모델이 단어(의미 단위)를 맞췄는지 측정합니다.
    """
    # 공백을 제거한 상태에서 형태소 분석을 수행하여 띄어쓰기 감점을 방지합니다.
    r_text = norm_ko(ref, remove_space=True)
    h_text = norm_ko(hyp, remove_space=True)

    r_morphs = mecab.morphs(r_text)
    h_morphs = mecab.morphs(h_text)

    if not r_morphs:
        return 0.0 if not h_morphs else 1.0

    # 형태소 리스트 간의 편집 거리를 계산합니다.
    return Levenshtein.distance(r_morphs, h_morphs) / len(r_morphs)

# # 1. 알파벳 발음 매핑 (영어가 포함된 경우 대비)
# ALPHABET_TO_KO = {
#     "a": "에이", "b": "비", "c": "씨", "d": "디", "e": "이", "f": "에프", "g": "지",
#     "h": "에이치", "i": "아이", "j": "제이", "k": "케이", "l": "엘", "m": "엠",
#     "n": "엔", "o": "오", "p": "피", "q": "큐", "r": "알", "s": "에스",
#     "t": "티", "u": "유", "v": "브이", "w": "더블유", "x": "엑스", "y": "와이", "z": "제트"
# }

# # 2. TV 도메인 전용 영어 단어 발음 매핑
# ENG_WORD_TO_KO = {
#     "netflix": "넷플릭스", "youtube": "유튜브", "disney": "디즈니",
#     "tving": "티빙", "apple": "애플", "plus": "플러스", "tv": "티비"
# }

# # 3. 한글 수사 통일 (하나 -> 일, 둘 -> 이 등)
# NUM_SENSE_MAP = {
#     "하나": "일", "한": "일", "둘": "이", "두": "이",
#     "셋": "삼", "세": "삼", "넷": "사", "네": "사",
#     "여덟": "팔", "열": "십"
# }

# def num_to_ko(num_str: str) -> str:
#     """숫자를 한국어 읽기 방식(0~999)으로 변환"""
#     try:
#         n = int(num_str)
#         if n == 0: return "영"
#         units = ["", "일", "이", "삼", "사", "오", "육", "칠", "팔", "구"]
#         tens = ["", "십", "이십", "삼십", "사십", "오십", "육십", "칠십", "팔십", "구십"]
#         hundreds = ["", "백", "이백", "삼백", "사백", "오백", "육백", "칠백", "팔백", "구백"]

#         if n >= 100:
#             h_val, rest = divmod(n, 100)
#             t_val, u_val = divmod(rest, 10)
#             return hundreds[h_val] + (tens[t_val] if t_val != 1 else "십") + units[u_val]
#         elif n >= 10:
#             t_val, u_val = divmod(n, 10)
#             return (tens[t_val] if t_val != 1 else "십") + units[u_val]
#         else:
#             return units[n]
#     except:
#         return num_str

# def norm_ko(s: str) -> str:
#     """
#     [강화된 정규화] 모든 숫자, 영어, 수사를 한글 발음으로 변환 후 공백 제거
#     """
#     if not s: return ""

#     # 소문자 변환 및 양끝 공백 제거
#     s = s.strip().lower()

#     # (1) 도메인 특화 영어 단어 변환 (netflix -> 넷플릭스)
#     for eng, ko in ENG_WORD_TO_KO.items():
#         s = s.replace(eng, ko)

#     # (2) 한글 수사 통일 (여덟 -> 팔)
#     for k, v in NUM_SENSE_MAP.items():
#         s = s.replace(k, v)

#     # (3) 숫자 -> 한글 발음 변환 (8 -> 팔)
#     s = re.sub(r'\d+', lambda m: num_to_ko(m.group()), s)

#     # (4) 남은 영어 알파벳 변환 (mbc -> 엠비씨)
#     temp_s = ""
#     for char in s:
#         temp_s += ALPHABET_TO_KO.get(char, char)
#     s = temp_s

#     # (5) 한글, 숫자 외 특수문자 제거 및 공백 제거
#     s = re.sub(r"[^0-9a-z\uac00-\ud7a3]", "", s)

#     return s

def best_substring_cer(entity: str, hyp: str, tol: int = 2) -> Tuple[float, str]:
    """
    [개선] 핵심어와 가장 유사한 부분 문자열을 찾아 오차율과 해당 문자열을 반환합니다.
    """
    e = norm_ko(entity)
    h = norm_ko(hyp).replace(" ","")
    if not e: return 0.0, ""
    if not h: return 1.0, ""

    L = len(e)
    # 문장이 핵심어보다 짧으면 전체를 비교 대상으로 설정
    if len(h) <= L:
        return Levenshtein.distance(e, h) / L, h

    best_score = 1.0
    best_sub = ""

    # Sliding Window 방식으로 최적의 매칭 구간 탐색
    for wlen in range(max(1, L - tol), min(len(h), L + tol) + 1):
        for i in range(0, len(h) - wlen + 1):
            sub = h[i:i + wlen]
            score = Levenshtein.distance(e, sub) / L
            if score < best_score:
                best_score = score
                best_sub = sub
                if best_score == 0.0:
                    return 0.0, best_sub

    return best_score, best_sub

class BiasManager:
    def __init__(self, path: str):
        self.path = path
        with open(path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        # 1. 프로그램 실행 시마다 참조 횟수 즉시 증가
        self.ref_count = self.data["ref_count"]
        self.data["ref_count"]+=1

        # 2. 이번 실행 세션 동안 인식된 고유명사를 저장할 메모리 딕셔너리
        self.session_hits = {}

    def get_weighted_hotwords(self, top_k: int) -> List[str]:
        """현재 JSON 내 가중치를 반영하여 랜덤하게 hotwords 선택 (Faster-Whisper용)"""
        if top_k <= 0: return []

        words = list(self.data["global"].keys())
        # 확률 보정 (가중치 + 1) [cite: 24, 54]
        weights = [float(self.data["global"][w]) + 1.0 for w in words]

        return random.choices(words, weights=weights, k=min(top_k, len(words)))

    def add_hit(self, matched_entities: List[str]):
        """인식된 고유명사 횟수를 메모리에만 임시 저장 (속도 저하 방지)"""
        for ent in matched_entities:
            self.session_hits[ent] = self.session_hits.get(ent, 0) + 1

    def finalize(self):
        """프로그램 종료 시 호출: 참조 횟수가 10의 배수일 때만 가중치 업데이트 및 파일 저장"""
        # 10의 배수 확인
        if self.ref_count != -3:
            print(f"\n[SYSTEM] 10회 주기 도달 (현재 {self.ref_count}회). 가중치 업데이트를 시작합니다.")
            # 전체 리스트를 탐색하는 대신, 이번 세션에 인식된 단어(hits)만 업데이트하여 속도 최적화
            for word, count in self.session_hits.items():
                if word in self.data["global"]:
                    # self.data["global"][word] += count
                    self.data["global"][word] +=count

            # 가중치 업데이트 내용 저장
            with open(self.path, "w", encoding="utf-8") as f:
                json.dump(self.data, f, ensure_ascii=False, indent=2)
            print("[SYSTEM] 가중치 및 참조 횟수 저장 완료.")
        else:
            # 10의 배수가 아닐 때는 참조 횟수만 업데이트하여 저장
            with open(self.path, "w", encoding="utf-8") as f:
                json.dump(self.data, f, ensure_ascii=False, indent=2)
            print(f"\n[SYSTEM] 참조 횟수 기록 완료 (현재 {self.ref_count}회). 가중치는 다음 10의 배수 실행 시 업데이트됩니다.")

def proper_metrics(entities: List[str], hyp: str, th: float = 0.2) -> Tuple[Optional[float], Optional[float], List[float], List[str]]:
    """
    [개선] 고유명사 성능 측정 및 인식된 단어 리스트 추출
    - return: recall, pn_cer, cers_list, matched_entities_list
    """
    if not entities:
        return None, None, []

    # 각 엔티티별로 점수와 매칭된 문자열을 가져옴
    results = [best_substring_cer(e, hyp) for e in entities]
    cers = [res[0] for res in results]
    matched_texts = [res[1].replace(" ", "") for res in results]

    recall = sum(1 for c in cers if c <= th) / len(cers)
    pn_cer = sum(cers) / len(cers)

    return recall, pn_cer, matched_texts


# -----------------------------
# 2) 데이터 로드 (Input)
# -----------------------------

def load_json(path: str) -> Dict[str, Any]:
    """JSON 파일 읽기 유틸리티"""
    if not path or not os.path.isfile(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_transcripts(path: str) -> Dict[str, Dict[str, Any]]:
    """
    [정답지 로드] transcripts.json 파싱
    - 구조: {"파일명.wav": {"text": "정답문장", "entities": ["키워드"]}
    """
    data = load_json(path)
    out: Dict[str, Dict[str, Any]] = {}
    for k, v in (data or {}).items():
        if isinstance(v, str):
            out[k] = {"text": v, "entities": []}
        elif isinstance(v, dict):
            out[k] = {
                "text": (v.get("text") or "").strip(),
                "entities": v.get("entities", []) or []
            }
        else:
            out[k] = {"text": "", "entities": []}

    return out


# -----------------------------
# 3) EPG/Catalog + Hotwords pool 구성
# -----------------------------
# 설명: ASR 모델에게 "이 단어들이 나올 확률이 높아"라고 힌트(Hotwords)를 주기 위한 데이터 준비 단계

def dedup_clean(words: List[str]) -> List[str]:
    """리스트 내 중복 제거 및 짧은 단어 필터링"""
    seen, out = set(), []
    for w in words or []:
        w = (w or "").strip()
        if not w or len(w) < 2: continue
        if w in seen: continue
        seen.add(w)
        out.append(w)
    return out

def epg_titles(epg: Dict[str, Any]) -> List[str]:
    """방송 편성표(EPG)에서 현재/오늘 방송 제목 추출"""
    titles = []
    for section in ("now", "today"):
        for item in epg.get(section, []) or []:
            if isinstance(item, dict) and item.get("title"):
                titles.append(item["title"])
            elif isinstance(item, str):
                titles.append(item)
    return dedup_clean(titles)

def catalog_titles(catalog: Dict[str, Any]) -> List[str]:
    """VOD 카탈로그에서 영화/드라마 제목 추출"""
    if not isinstance(catalog, dict) or not catalog: return []
    if isinstance(catalog.get("titles"), list):
        return dedup_clean([x for x in catalog["titles"] if isinstance(x, str)])
    out = []
    for _, v in catalog.items():
        if isinstance(v, list): out += [x for x in v if isinstance(x, str)]
    return dedup_clean(out)

def coerce_terms(v: Any) -> List[str]:
    """Biasing List(가중치 사전) 포맷을 단순 리스트로 변환"""
    if not v: return []
    if isinstance(v, list) and all(isinstance(x, str) for x in v):
        return [x.strip() for x in v if x and len(x.strip()) >= 2]
    if isinstance(v, list) and all(isinstance(x, dict) for x in v):
        items = []
        for d in v:
            term = (d.get("term") or "").strip()
            score = d.get("score", 0)
            if term and len(term) >= 2:
                items.append((term, score))
        items.sort(key=lambda t: t[1], reverse=True)
        return [t for t, _ in items]
    return []


# -----------------------------
# 4) 최소 NLU (규칙 기반 Intent/Slot 파서)
# -----------------------------
# 설명: 복잡한 딥러닝 모델 대신, 특정 단어가 포함되면 의도를 추론하는 'Rule-based' 방식.
# LLM 없이도 기본적인 TV 제어는 가능하다는 것을 보여주는 부분.

# INTENTS = {
#     "POWER_OFF": "전원 끄기", "POWER_ON": "전원 켜기",
#     "VOLUME_SET": "볼륨 설정", "VOLUME_UP": "볼륨 올리기", "VOLUME_DOWN": "볼륨 내리기",
#     "CHANNEL_TUNE": "채널 변경", "APP_OPEN": "앱 실행", "CONTENT_PLAY": "콘텐츠 재생",
#     "UNKNOWN": "미분류",
# }

# # 앱 이름 동의어 사전 (유저가 '유튭'이라 해도 '유튜브'로 인식하기 위함)
# APP_ALIASES = {
#     "유튜브": ["유튜브", "youtube", "you tube"],
#     "넷플릭스": ["넷플릭스", "netflix"],
#     "디즈니플러스": ["디즈니플러스", "disney plus", "disney+", "디즈니+"],
#     "티빙": ["티빙", "tving", "tv ing", "tv-ing"],
# }

# # 채널명 동의어 사전
# CHANNEL_ALIASES = {
#     "MBC":  ["엠비씨", "mbc", "엠비시"],
#     "SBS":  ["에스비에스", "sbs"],
#     "JTBC": ["제이티비씨", "jtbc"],
#     "KBS1": ["케이비에스1", "kbs1", "케이비에스 1", "kbs 1"],
#     "KBS2": ["케이비에스2", "kbs2", "케이비에스 2", "kbs 2"],
#     "TVN":  ["티비엔", "tvn", "티브이엔", "tv n"],
#     "YTN":  ["와이티엔", "ytn"],
# }


# def channel_terms_for_hotwords() -> List[str]:
#     """Hotwords(힌트)에 채널명들도 추가하여 인식률 향상"""
#     terms: List[str] = []
#     for ch_id, keys in (CHANNEL_ALIASES or {}).items():
#         if ch_id and isinstance(ch_id, str):
#             terms.append(ch_id)
#         for k in keys or []:
#             if isinstance(k, str) and k.strip():
#                 terms.append(k.strip())
#     return dedup_clean(terms)


def _contains_any(s: str, keys: List[str]) -> bool:
    """문자열 s 안에 keys 중 하나라도 포함되어 있는지 확인"""
    sl = (s or "").lower()
    return any(k.lower() in sl for k in keys)

# def extract_app(text: str) -> Optional[str]:
#     """텍스트에서 앱 이름 추출 (동의어 처리 포함)"""
#     if not text: return None
#     for app, keys in APP_ALIASES.items():
#         if _contains_any(text, keys):
#             return app
#     return None

# def extract_channel(text: str) -> Optional[str]:
#     """텍스트에서 채널명 추출 (정규화 포함)"""
#     if not text: return None
#     tl = (text or "").lower()
#     tl = re.sub(r"\s+", " ", tl).strip()
#     tl_ns = tl.replace(" ", "")

#     for ch_id, keys in CHANNEL_ALIASES.items():
#         for k in keys:
#             k_l = k.lower()
#             # 띄어쓰기 있는 버전과 없는 버전 모두 검사
#             if k_l in tl or k_l.replace(" ", "") in tl_ns:
#                 return ch_id
#     return None

def extract_int_number_0_10(text: str) -> Optional[int]:
    """
    [볼륨 추출] 텍스트에서 0~10 사이의 숫자를 찾음
    - '볼륨 5로 해줘', '볼륨 다섯' 등을 모두 처리
    """
    if not text:
        return None

    # 1. 아라비아 숫자 찾기 (정규식)
    m = re.search(r"(?<!\d)(\d{1,2})(?!\d)", text)
    if m:
        try:
            n = int(m.group(1))
            if 0 <= n <= 10:
                return n
        except:
            pass

    # 2. 한글 숫자 찾기 (매핑 테이블 활용)
    t = re.sub(r"\s+", "", text)
    for k, v in NUM_KO_0_10.items():
        if k in t:
            return v
    return None

# def nlu_parse(text: str, catalog_pool: List[str]) -> Dict[str, Any]:
#     """
#     [NLU 핵심 로직] 텍스트를 입력받아 Intent(의도)와 Slot(상세정보) 반환
#     - 우선순위: 전원 > 볼륨 > 채널 > 앱 > 콘텐츠 검색 순으로 검사
#     """
#     out = {"intent": "UNKNOWN", "slots": {}}
#     if not text: return out
#     tl = text.lower()

#     # 1) 전원 제어
#     if ("꺼" in tl or "끄" in tl) and ("tv" in tl or "티비" in tl or "전원" in tl):
#         out["intent"] = "POWER_OFF"
#         return out
#     if ("켜" in tl or "켜줘" in tl) and ("tv" in tl or "티비" in tl or "전원" in tl):
#         out["intent"] = "POWER_ON"
#         return out

#     # 2) 볼륨 제어
#     if "볼륨" in tl or "volume" in tl:
#         n = extract_int_number_0_10(text)
#         if n is not None:
#             out["intent"] = "VOLUME_SET"
#             out["slots"]["volume"] = int(n)
#             return out
#         if "올" in tl or "키" in tl or "높" in tl:
#             out["intent"] = "VOLUME_UP"
#             return out
#         if "내" in tl or "줄" in tl or "낮" in tl:
#             out["intent"] = "VOLUME_DOWN"
#             return out

#     # 3) 채널 제어
#     ch = extract_channel(text)
#     if ch:
#         # 채널명이 있다고 무조건 채널 변경이 아님 (예: "MBC에서 하는 드라마 찾아줘")
#         # 따라서 명령어가 같이 있어야 함.
#         if ("채널" in tl) or ("바꿔" in tl) or ("변경" in tl) or ("틀" in tl) or ("켜" in tl) or ("봐" in tl) or ("보여" in tl):
#             out["intent"] = "CHANNEL_TUNE"
#             out["slots"]["channel_id"] = ch
#             return out

#     # 4) 앱 실행
#     app = extract_app(text)
#     if app and ("켜" in tl or "열" in tl or "실행" in tl or "틀" in tl):
#         out["intent"] = "APP_OPEN"
#         out["slots"]["app"] = app
#         return out

#     # 5) 콘텐츠 검색 (Fuzzy Matching)
#     # ASR 결과가 정확하지 않아도 DB에 있는 제목과 가장 유사하면 검색 의도로 파악
#     if ("틀" in tl or "재생" in tl or "보여" in tl or "켜" in tl) and catalog_pool:
#         best = process.extractOne(text, catalog_pool, scorer=fuzz.WRatio)
#         if best:
#             title, score, _ = best
#             if score >= 75:  # 유사도가 75점 이상일 때만 인정
#                 out["intent"] = "CONTENT_PLAY"
#                 out["slots"]["title"] = title
#                 out["slots"]["title_score"] = float(score)
#                 return out

#     return out

# def intent_slot_metrics(ref_intent, ref_slots, hyp_intent, hyp_slots):
#     """NLU 결과 채점 (정답과 비교)"""
#     if not ref_intent: return None, None
#     intent_acc = 1 if (ref_intent == hyp_intent) else 0 # 의도 정확도

#     if not ref_slots or not isinstance(ref_slots, dict):
#         return intent_acc, None

#     keys = list(ref_slots.keys())
#     if not keys: return intent_acc, 1.0

#     # Slot 정확도 (예: 채널명은 맞췄는지, 볼륨 숫자는 맞췄는지)
#     ok = 0
#     for k in keys:
#         if k in hyp_slots and hyp_slots.get(k) == ref_slots.get(k):
#             ok += 1
#     return intent_acc, ok / len(keys)

# def hotwords_from_context(bias: Dict[str, Any], epg: Dict[str, Any], catalog: Dict[str, Any], top_k: int) -> List[str]:
#     """
#     [Context Injection] 현재 상황에 맞춰 ASR에게 힌트로 줄 단어장 생성
#     - 편성표 제목, VOD 제목, 채널명 등을 모아서 top_k개 만큼 추림.
#     """
#     if top_k <= 0: return []
#     pool = dedup_clean(
#         epg_titles(epg)
#         + coerce_terms(bias.get("global"))
#         + catalog_titles(catalog)
#         + channel_terms_for_hotwords()
#     )
#     return pool[:top_k]

def hotwords_from_context(bias: Dict[str, Any], epg: Dict[str, Any], catalog: Dict[str, Any], top_k: int) -> List[str]:
    """
    [Context Injection] 현재 상황에 맞춰 ASR에게 힌트로 줄 단어장 생성
    - 전체 리스트에서 중복을 제거한 후, 랜덤하게 top_k개를 선정하여 반환합니다.
    """
    if top_k <= 0:
        return []

    # 1. 전체 후보 단어 풀 생성 (중복 제거 포함)
    pool = dedup_clean(
        epg_titles(epg)
        + coerce_terms(bias.get("global"))
        + catalog_titles(catalog)
    )

    # 2. 리스트보다 요청한 개수(top_k)가 많을 경우를 대비한 예외 처리
    actual_k = min(top_k, len(pool))

    # 3. 랜덤 샘플링 수행 (순서 섞임 효과 포함)
    return random.sample(pool, actual_k)


# -----------------------------
# 5~6) 후처리 (Whitelist Correction)
# -----------------------------
# 보고서 포인트: ASR의 한계를 극복하는 핵심 기술.
# ASR이 "무한 도잔 틀어줘"라고 했을 때 DB의 "무한도전"으로 고쳐주는 로직.

def make_whitelist(hyp_raw: str, epg: Dict[str, Any], catalog: Dict[str, Any], top_n: int = 80) -> List[str]:
    """
    현재 인식된 문장(hyp_raw)과 가장 비슷한 DB 내의 제목 후보군(Top N)을 뽑음.
    전체 DB를 다 뒤지면 느리므로, 1차적으로 가능성 있는 것만 추림.
    """
    pool = dedup_clean(epg_titles(epg) + catalog_titles(catalog))
    if not pool: return []
    # RapidFuzz를 사용하여 빠르게 유사도 검색
    scored = process.extract(hyp_raw, pool, scorer=fuzz.WRatio, limit=min(top_n, len(pool)))
    return [t for t, _, _ in scored]

def build_norm_with_map(raw: str) -> Tuple[str, List[int]]:
    """원본 문자열의 인덱스를 보존하기 위한 유틸리티 (후처리 교체 위치 계산용)"""
    if not raw: return "", []
    raw_l = raw.lower()
    norm_chars, idx_map = [], []
    for i, ch in enumerate(raw_l):
        if re.match(r"[0-9a-z\u3131-\u318e\uac00-\ud7a3]", ch):
            norm_chars.append(ch)
            idx_map.append(i) # 정규화된 문자가 원본의 몇 번째 문자인지 저장
    return "".join(norm_chars), idx_map

def best_substring_span_raw(entity: str, hyp_raw: str, tol: int = 2) -> Tuple[Optional[int], Optional[int], float]:
    """
    [교정 위치 탐색] '무한도잔 틀어줘'에서 '무한도잔'이 어디서부터 어디까지인지(Index) 찾음.
    - 정규화된 문자열에서 위치를 찾은 뒤, `build_norm_with_map`을 통해 원본 문자열 인덱스로 변환.
    """
    e = norm_ko(entity)
    if not e: return None, None, 1.0
    h_norm, h_map = build_norm_with_map(hyp_raw)
    if not h_norm: return None, None, 1.0

    L = len(e)
    # 문장이 너무 짧으면 전체 비교
    if len(h_norm) <= L:
        span_cer = Levenshtein.distance(e, h_norm) / L
        s_raw = h_map[0] if h_map else 0
        e_raw = (h_map[-1] + 1) if h_map else len(hyp_raw)
        return s_raw, e_raw, span_cer

    # 부분 문자열 탐색
    best = 1.0
    best_s_norm = 0
    best_e_norm = min(len(h_norm), L)

    for wlen in range(max(1, L - tol), min(len(h_norm), L + tol) + 1):
        for i in range(0, len(h_norm) - wlen + 1):
            sub = h_norm[i:i + wlen]
            span_cer = Levenshtein.distance(e, sub) / L
            if span_cer < best:
                best = span_cer
                best_s_norm = i
                best_e_norm = i + wlen
                if best == 0.0: break
        if best == 0.0: break

    # 찾은 위치를 원본 인덱스로 변환
    s_raw = h_map[best_s_norm]
    e_raw = h_map[best_e_norm - 1] + 1
    return s_raw, e_raw, best

def postprocess_rule_whitelist(
    hyp_raw: str,
    whitelist: List[str],
    wratio_th: int = 92,
    gate: float = 0.34,
    tol: int = 2
) -> Tuple[str, List[Dict[str, Any]]]:
    """
    [후처리 메인 로직] Gate Mechanism 적용
    - 기능: ASR 결과(hyp_raw)를 화이트리스트에 있는 정확한 제목으로 치환.
    - Gate 1 (유사도): 전체 문장과 제목이 너무 다르면(WRatio < 92) 교정 안 함.
    - Gate 2 (부분오차): 교체하려는 부분만 봤을 때도 너무 다르면(CER > 0.34) 교정 안 함.
    """
    if not hyp_raw or not whitelist:
        return hyp_raw, []

    # 1. 명령어 제거 후 순수 제목(target) 추출 시도
    # 예: "무한도잔 틀어줘" -> "무한도잔"
    target = re.sub(
        r"(틀어(줘)?|재생(해(줘)?)?|보여(줘)?|켜(줘)?|해(줘)?|바꿔(줘)?|변경(해(줘)?)?|채널|좀|지금|다시|tv|티비|전원)",
        " ",
        (hyp_raw or "").lower()
    )
    target = re.sub(r"\s+", " ", target).strip()

    # 2. 가장 유사한 후보 찾기
    best = process.extractOne(target if len(target) >= 2 else hyp_raw, whitelist, scorer=fuzz.WRatio)
    if not best:
        return hyp_raw, []
    chosen, wr_score, _ = best

    # Gate 1: 전체 유사도 체크 (임계치 미만이면 교정 포기)
    if wr_score < wratio_th:
        return hyp_raw, [{
            "type": "skip",
            "reason": "wratio<th",
            "wratio": float(wr_score),
            "chosen": chosen,
            "target": target
        }]

    # Gate 2: 부분 교체 구간 탐색 및 CER 체크
    s, e, span_cer = best_substring_span_raw(chosen, hyp_raw, tol=tol)
    if s is None or e is None or span_cer > gate:
        return hyp_raw, [{
            "type": "skip",
            "reason": "gate_fail",
            "wratio": float(wr_score),
            "chosen": chosen,
            "span_cer": float(span_cer),
            "target": target
        }]

    # 이미 정확하면 패스
    surface = hyp_raw[s:e]
    if surface == chosen:
        return hyp_raw, [{
            "type": "noop",
            "wratio": float(wr_score),
            "chosen": chosen,
            "span_cer": float(span_cer),
            "target": target
        }]

    # 3. 최종 교체 실행 (String Replacement)
    hyp_pp = hyp_raw[:s] + chosen + hyp_raw[e:]
    replog = [{
        "type": "replace",
        "start": int(s),
        "end": int(e),
        "from": surface,
        "to": chosen,
        "wratio": float(wr_score),
        "span_cer": float(span_cer),
        "target": target
    }]
    return hyp_pp, replog


# -----------------------------
# 7) ASR 래퍼 (Whisper 실행)
# -----------------------------

@dataclass
class ASRConfig:
    model_size: str = "medium"
    device: str = "cuda"
    compute_type: str = "float16"
    language: str = "ko"
    beam_size: int = 5

class ASR:
    """Faster-Whisper 모델을 관리하고 추론하는 클래스"""
    def __init__(self, cfg: ASRConfig):
        self.cfg = cfg
        # 모델 로드 (최초 1회 시간이 걸림)
        self.model = WhisperModel(cfg.model_size, device=cfg.device, compute_type=cfg.compute_type)

    def transcribe(self, path: str, hotwords: Optional[List[str]] = None) -> str:
        """
        [음성 인식 실행]
        - path: 오디오 파일 경로
        - hotwords: 우선적으로 인식할 단어 리스트 (Prompting)
        """
        # 기존의 너무 긴 프롬프트를 핵심 요약 버전으로 교체
        korean_only_prompt = (
            "엠비씨 뉴스데스크, 티브이엔 유퀴즈, 넷플릭스 파친코 틀어줘, "
            "볼륨 십으로 올려줘, 삼십 분 뒤에 티비 꺼줘, 채널 이십이 번으로 바꿔줘, "
            "에이, 비, 씨, 디, 하나, 둘, 셋, 삼십 초, 오 분, 세 칸, 일 배속"
        )
        kwargs: Dict[str, Any] = {"language": self.cfg.language, "beam_size": self.cfg.beam_size, "initial_prompt":korean_only_prompt, "vad_filter":True}
        # hotwords가 리스트 형태로 들어왔을 때만 처리
        if hotwords and len(hotwords) > 0:
          kwargs["hotwords"] = ",".join(hotwords)
        # 실제 추론 발생s
        segs, _ = self.model.transcribe(path, **kwargs)
        return "".join(s.text for s in segs).strip()


# -----------------------------
# 8) 결과 요약 (Report)
# -----------------------------

def summarize(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    [실험 결과 집계]
    - 모든 개별 파일의 결과를 모아서 'Hotwords 개수'와 '후처리 여부'에 따른 평균 점수를 냄.
    """
    agg: Dict[Tuple[int, int], Dict[str, Any]] = {}
    for r in rows:
        k = int(r["top_k"])
        pp = int(r["preprocess_on"])
        key = (k, pp)

        a = agg.setdefault(
            key,
            {
                "top_k": k, "preprocess_on": pp, "files_num": 0,
                "cer_sum": 0.0, "wer_sum": 0.0, "pn_r_sum": 0.0, "pn_c_sum": 0.0, "pn_n": 0,
                "intent_sum": 0.0, "intent_n": 0, "slot_sum": 0.0, "slot_n": 0,
            }
        )
        a["files_num"] += 1
        a["cer_sum"] += float(r["cer"])
        a["wer_sum"] += float(r["wer"])

        if r.get("pn_recall") is not None and r.get("pn_cer") is not None:
            a["pn_r_sum"] += float(r["pn_recall"])
            a["pn_c_sum"] += float(r["pn_cer"])
            a["pn_n"] += 1

        if r.get("intent_acc") is not None:
            a["intent_sum"] += float(r["intent_acc"])
            a["intent_n"] += 1

        if r.get("slot_acc") is not None:
            a["slot_sum"] += float(r["slot_acc"])
            a["slot_n"] += 1

    # 평균 계산
    out = []
    for (k, pp) in sorted(agg.keys()):
        a = agg[(k, pp)]
        pn_n = a["pn_n"]
        out.append({
            "top_k": k,         # Hotwords 개수
            "preprocess_on": pp,        # 후처리 여부 (0/1)
            "files_num": a["files_num"],        # 테스트 샘플 수
            "cer_mean": a["cer_sum"] / max(1, a["files_num"]), # 평균 오타율
            "wer_mean": a["wer_sum"] / max(1, a["files_num"]), # 평균 자음 오타율
            "pn_recall_mean": (a["pn_r_sum"] / pn_n) if pn_n else None, # 고유명사 인식 성공률
            "pn_cer_mean": (a["pn_c_sum"] / pn_n) if pn_n else None,    # 고유명사 오타율
            "intent_acc_mean": (a["intent_sum"] / a["intent_n"]) if a["intent_n"] else None, # 의도 분류 정확도
            "slot_acc_mean": (a["slot_sum"] / a["slot_n"]) if a["slot_n"] else None,         # 슬롯 추출 정확도
        })
    return out


# -----------------------------
# 9) 메인 실행부 (Execution)
# -----------------------------

# 1. 파일 존재 여부 검증
assert os.path.isdir(AUDIO_FOLDER), f"오디오 폴더 없음: {AUDIO_FOLDER}"
assert os.path.isfile(TRANSCRIPTS_PATH), f"transcripts.json 없음: {TRANSCRIPTS_PATH}"

# 2. 데이터 로딩
transcripts = load_transcripts(TRANSCRIPTS_PATH)
bias = load_json(BIAS_PATH)
epg = load_json(EPG_PATH)
catalog = load_json(CATALOG_PATH)

if not epg:
    print("[WARN] epg.json이 없거나 비어있습니다. hotwords 성능이 떨어질 수 있습니다.")
if not catalog:
    print("[WARN] catalog.json이 없거나 비어있습니다.")

files = glob.glob(os.path.join(AUDIO_FOLDER, "**/*.mp4"), recursive=True)
assert files, f"오디오 없음: {AUDIO_FOLDER}"

# 3. 모델 초기화
cfg = ASRConfig(model_size=ASR_MODEL, device=ASR_DEVICE, compute_type=ASR_COMPUTE, language=ASR_LANG, beam_size=ASR_BEAM)
asr = ASR(cfg)

g2p=G2p()

mecab=MeCab()

# NLU용 콘텐츠 풀 (편성표 + 카탈로그 합본)
content_pool = dedup_clean(epg_titles(epg) + catalog_titles(catalog))
rows: List[Dict[str, Any]] = []

#hotwords 편향치 업데이트용 객
bias_mgr = BiasManager(BIAS_PATH)

# 4. 실험 루프 (Grid Search)
# TOPK_SWEEP: Hotwords 개수를 바꿔가며 테스트
for top_k in TOPK_SWEEP:
    # 현재 상황에 맞는 힌트 단어(Hotwords) 생성

    for repeat in range(REPEAT_NUM):
          #hotwords 편향치 업데이트
      bias_mgr.finalize()
      hotwords = bias_mgr.get_weighted_hotwords(top_k)

      # POSTPROCESS_SWEEP: 후처리 ON/OFF 테스트
      for pp_on in POSTPROCESS_SWEEP:
          print(f"\n[RUN] top_k={top_k} | postprocess_on={pp_on} | Gate(WRatio={RULE_WRATIO_TH}) | Gate={RULE_GATE}\nhotwords: {hotwords}")

          for ap in files:
              print(ap)
              fname = os.path.basename(ap)
              meta = transcripts.get(fname)
              if not meta:
                  continue

              # 정답 데이터 추출
              ref = meta.get("text", "")
              ents = meta.get("entities", []) or []


              # A) ASR 실행 (듣기)
              hyp_raw = asr.transcribe(ap, hotwords=hotwords)

              # B) Whitelist 후보군 탐색 (교정 후보 찾기)
              wl = make_whitelist(hyp_raw, epg=epg, catalog=catalog, top_n=WL_TOPN)
              wl_top5 = wl[:5]

              # C) Post-processing (고치기)
              if pp_on and wl:
                  hyp_pp, replog = postprocess_rule_whitelist(
                      hyp_raw, wl,
                      wratio_th=RULE_WRATIO_TH,
                      gate=RULE_GATE,
                      tol=RULE_TOL
                  )
              else:
                  hyp_pp, replog = hyp_raw, []

              # D) Metrics (채점하기 - 텍스트 정확도)
              cer = cer_norm(ref, hyp_pp)
              wer=wer_norm(ref,hyp_pp)
              pn_r, pn_c, hyp_ents = proper_metrics(ents, hyp_pp)

              #인식된 고유명사 count 추가
              bias_mgr.add_hit(hyp_ents)

              # # E) NLU + Metrics (이해하기 & 채점)
              # nlu = {"intent": None, "slots": {}}
              # intent_acc, slot_acc = None, None
              # if ENABLE_NLU:
              #     nlu = nlu_parse(hyp_pp, content_pool)
              #     intent_acc, slot_acc = intent_slot_metrics(ref_intent, ref_slots, nlu["intent"], nlu.get("slots", {}))

              # 수정 후: pn_r이 None이면 0.0으로 출력하게끔 변경
              pn_r= pn_r if pn_r is not None else 0.0

              # 로그 출력
              print(f"- {os.path.dirname(ap).split("/")[-1]}/{fname} | pp_on={pp_on} | cer={cer:.4f} | wer={wer:.4f} | pn_c={pn_c} | pn_recall={pn_r:.4f}")
              print(f"ref: {ref}\nhyp_raw: {norm_ko(hyp_raw)}\nref_pp: {ents}\nhyp_pp: {hyp_ents}\n")
              # 결과 행 저장
              rows.append({
                  "file": f'{os.path.dirname(ap).split("/")[-1]}/{fname}',
                  "top_k": top_k,
                  "hotwords":hotwords,
                  "preprocess_on": pp_on,
                  "cer": cer,
                  "wer":wer,
                  "pn_recall": pn_r,
                  "pn_cer": pn_c,
                  "hotwords_n": len(hotwords),
                  "wl_size": len(wl),
                  "wl_top5": json.dumps(wl_top5, ensure_ascii=False),
                  "ref": ref,
                  "hyp_raw": norm_ko(hyp_raw),
                  "ref_pp": ents,
                  "hyp_pp": hyp_ents,
                  "replog": json.dumps(replog, ensure_ascii=False),
              })

# 5. CSV 파일 저장
assert rows, "rows가 비었습니다. transcripts.json의 파일명과 AUDIO_FOLDER 파일명이 일치하는지 확인하세요."

#hotwords 편향치 업데이트
bias_mgr.finalize()

# 상세 결과 저장
with open(OUT_ROWS, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

# 요약 결과 저장
summary = summarize(rows)
with open(OUT_SUM, "w", newline="", encoding="utf-8-sig") as f:
    fields = list(summary[0].keys()) if summary else ["top_k", "pp_on", "n"]
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(summary)
print(summary)


print("\n[DONE] 실행 완료")
print("- 상세 결과: ", OUT_ROWS)
print("- 요약 결과: ", OUT_SUM)
print("-> 두 CSV 파일을 다운로드하여 보고서에 활용하세요.")


[SYSTEM] 10회 주기 도달 (현재 5회). 가중치 업데이트를 시작합니다.
[SYSTEM] 가중치 및 참조 횟수 저장 완료.

[RUN] top_k=20 | postprocess_on=0 | Gate(WRatio=92) | Gate=0.34
hotwords: ['더문', '티빙', '임영웅', '유퀴즈온더블럭', '유튜브', '박찬욱', '언더커버하이이스쿨', '송중기', '엠비씨', '제이티비씨', '김계란', '범죄도시', '애니맥스', '조나단', '티빙', '김고은', '위키드', '낮과밤이다른그녀', '티빙', '애플티비플러스']
/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/voice_record/seowonryeol/record14.mp4
- seowonryeol/record14.mp4 | pp_on=0 | cer=0.0000 | wer=0.0000 | pn_c=None | pn_recall=0.0000
ref: 볼륨을 십으로 맞추고 화면 모드를 영화 모드로 변경해줘
hyp_raw: 볼륨을 십으로 맞추고 화면 모드를 영화 모드로 변경해줘
ref_pp: []
hyp_pp: []

/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/voice_record/seowonryeol/record11.mp4
- seowonryeol/record11.mp4 | pp_on=0 | cer=0.0909 | wer=0.0769 | pn_c=0.25 | pn_recall=0.5000
ref: 티브이엔 유퀴즈온더블럭 재방송 언제 하는지 찾아줘
hyp_raw: 티비엔 유퀴즈 온더블럭 재방송 언제 하는지 찾아줘
ref_pp: ['티브이엔', '유퀴즈온더블럭']
hyp_pp: ['티비엔', '유퀴즈온더블럭']

/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/voic

KeyboardInterrupt: 